In [5]:
import google.generativeai as genai
import pandas as pd
import json
import time
import math
import os
import re
from tqdm.notebook import tqdm
from dotenv import load_dotenv

# --- 1. SETUP & CREDENTIALS ---
load_dotenv()
API_KEY = os.getenv("GEMINI_API_KEY")

if not API_KEY:
    raise ValueError("API Key not found! Check your .env file.")

genai.configure(api_key=API_KEY)

# --- CONFIGURATION ---
INPUT_PATH = r"D:\PythonEnv\Interpretable\hs_sample_400\GPTResult\traindataset_p_00_sampled400_EXTREME_ZERO_TOL.csv"
OUTPUT_DIR_NAME = "gemini_res_1"
OUTPUT_FILE_NAME = "traindataset_p_00_sampled400_gemini_result.csv"
BATCH_SIZE = 50

# UPDATED: Using the high-limit model from your screenshot
MODEL_NAME = "gemini-2.5-flash-lite"

model = genai.GenerativeModel(
    model_name=MODEL_NAME,
    generation_config={
        "response_mime_type": "application/json",
        "temperature": 0.0,
    }
)

# Disable safety settings to allow hate speech detection
safety_settings = [
    {"category": "HARM_CATEGORY_HARASSMENT", "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_HATE_SPEECH", "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_SEXUALLY_EXPLICIT", "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_NONE"},
]

SYSTEM_PROMPT = """
You are a content moderation expert. Analyze the list of texts for Hate Speech.
Input: A list of text items.
Task: Determine if each item is Hate Speech (1) or Not Hate Speech (0).

CRITERIA:
1. No Respect
2. Insult
3. Dehumanize
4. Humiliate
5. Violence
6. Genocide
7. Attack/Defend
8. Status
9. Negative Sentiment
10. Hate Speech Score

INTERNAL JUDGMENT:
If any of the above 10 are true, result is 1.
If text contains slang/emojis implies hate speech, result 1.

OUTPUT:
Return a JSON list of objects. Each object must have exactly one key: "prediction" (integer 0 or 1).
"""

def clean_json_response(response_text):
    text = re.sub(r"```json\s*", "", response_text)
    text = re.sub(r"```\s*$", "", text)
    return text.strip()

def classify_batch_with_retry(texts, max_retries=5):
    """Sends a batch to Gemini with auto-retry for 429 errors."""

    prompt_content = "Classify these texts:\n"
    for i, text in enumerate(texts):
        clean_text = str(text).replace("\n", " ")
        prompt_content += f"ID {i}: {clean_text}\n"
    prompt_content += "\nReturn JSON list of objects with key 'prediction'."

    retries = 0
    while retries < max_retries:
        try:
            response = model.generate_content(
                [SYSTEM_PROMPT, prompt_content],
                safety_settings=safety_settings
            )
            cleaned_text = clean_json_response(response.text)
            return json.loads(cleaned_text)

        except Exception as e:
            error_str = str(e)

            # Check if it is a Rate Limit error (429 or Quota)
            if "429" in error_str or "quota" in error_str.lower():
                wait_time = 5 # Flash Lite is fast, short wait is usually fine
                print(f"\n⚠️ Quota hit! Sleeping for {wait_time}s...")
                time.sleep(wait_time)
                retries += 1
            else:
                print(f"\n❌ Non-retriable error: {e}")
                return [{"prediction": 0} for _ in texts]

    print("\n❌ Max retries reached. Skipping.")
    return [{"prediction": 0} for _ in texts]

# --- MAIN EXECUTION ---
if __name__ == "__main__":
    print(f"Reading file: {INPUT_PATH}")

    if not os.path.exists(INPUT_PATH):
        print("❌ Error: Input file not found.")
    else:
        try:
            df = pd.read_csv(INPUT_PATH, encoding='utf-8-sig')
        except UnicodeDecodeError:
            df = pd.read_csv(INPUT_PATH, encoding='latin1')

        base_dir = os.path.dirname(os.path.dirname(INPUT_PATH))
        output_dir = os.path.join(base_dir, OUTPUT_DIR_NAME)
        if not os.path.exists(output_dir):
            os.makedirs(output_dir)
        output_full_path = os.path.join(output_dir, OUTPUT_FILE_NAME)

        results = []
        print(f"Processing {len(df)} rows using model: {MODEL_NAME}...")

        pbar = tqdm(range(0, len(df), BATCH_SIZE))

        for i in pbar:
            batch = df.iloc[i : i + BATCH_SIZE]
            texts = batch['text'].tolist()

            batch_predictions = classify_batch_with_retry(texts)

            if len(batch_predictions) != len(texts):
                while len(batch_predictions) < len(texts):
                    batch_predictions.append({"prediction": 0})
                batch_predictions = batch_predictions[:len(texts)]

            for item in batch_predictions:
                val = item.get("prediction", 0)
                if val not in [0, 1]: val = 0
                results.append(val)

            # Flash Lite is so fast you barely need sleep, but 0.5s is safe
            time.sleep(0.5)

        df['hatespeech_LLM_gemini'] = results
        df.to_csv(output_full_path, index=False, encoding='utf-8-sig')
        print(f"\n✅ DONE! Saved to: {output_full_path}")

Reading file: D:\PythonEnv\Interpretable\hs_sample_400\GPTResult\traindataset_p_00_sampled400_EXTREME_ZERO_TOL.csv
Processing 400 rows using model: gemini-2.5-flash-lite...


  0%|          | 0/20 [00:00<?, ?it/s]


✅ DONE! Saved to: D:\PythonEnv\Interpretable\hs_sample_400\gemini_res\traindataset_p_00_sampled400_gemini_result.csv


In [7]:
import pandas as pd
import os

# --- CONFIGURATION ---
# Path to your file saved from the Gemini step
FILE_PATH = r"/hs_sample_400/gemini_res_1\traindataset_p_00_sampled400_gemini_result.csv"

def calculate_metrics(df, truth_col, pred_col, model_name, p1_truth, p0_truth):
    """
    Calculates P11, P00, and Pc based on the user's formula.
    """
    print(f"\n--- Analysis for {model_name} ---")

    # 1. P11 = P(Pred=1 | Truth=1) [Sensitivity / Recall]
    truth_1_subset = df[df[truth_col] == 1]
    if len(truth_1_subset) > 0:
        p11 = truth_1_subset[pred_col].mean()
    else:
        p11 = 0.0

    # 2. P00 = P(Pred=0 | Truth=0) [Specificity]
    truth_0_subset = df[df[truth_col] == 0]
    if len(truth_0_subset) > 0:
        # We want the fraction where Pred is 0.
        # (1 - mean) works because mean gives fraction of 1s.
        p00 = 1.0 - truth_0_subset[pred_col].mean()
    else:
        p00 = 0.0

    # 3. Pc = P1*P11 + P0*P00 [Overall Accuracy]
    # This represents the total probability of a correct classification
    pc = (p1_truth * p11) + (p0_truth * p00)

    print(f"P11 (Sensitivity): {p11:.4f}")
    print(f"P00 (Specificity): {p00:.4f}")
    print(f"Pc (Accuracy):     {pc:.4f}")

    return p11, p00, pc

def main():
    if not os.path.exists(FILE_PATH):
        print(f"Error: File not found at {FILE_PATH}")
        return

    try:
        df = pd.read_csv(FILE_PATH, encoding='utf-8-sig')
    except:
        df = pd.read_csv(FILE_PATH, encoding='latin1')

    # Ensure columns exist
    required = ['hs_state', 'hatespeech_llm_GPT', 'hatespeech_LLM_gemini']
    if not all(col in df.columns for col in required):
        print(f"Error: Missing columns. Found: {df.columns.tolist()}")
        return

    # --- Step 1: Calculate Priors (Ground Truth Probabilities) ---
    # P1 = P(Y=1)
    p1_truth = df['hs_state'].mean()
    # P0 = P(Y=0)
    p0_truth = 1.0 - p1_truth

    print(f"Ground Truth Statistics:")
    print(f"P1 (Prevalence of Hate Speech): {p1_truth:.4f}")
    print(f"P0 (Prevalence of Normal Text): {p0_truth:.4f}")

    # --- Step 2: Calculate Metrics for GPT ---
    p11_gpt, p00_gpt, pc_gpt = calculate_metrics(
        df,
        truth_col='hs_state',
        pred_col='hatespeech_llm_GPT',
        model_name='GPT',
        p1_truth=p1_truth,
        p0_truth=p0_truth
    )

    # --- Step 3: Calculate Metrics for Gemini ---
    p11_gem, p00_gem, pc_gem = calculate_metrics(
        df,
        truth_col='hs_state',
        pred_col='hatespeech_LLM_gemini',
        model_name='Gemini',
        p1_truth=p1_truth,
        p0_truth=p0_truth
    )

    # --- Final Comparison ---
    print("\n" + "="*50)
    print(f"{'Metric':<20} | {'GPT':<10} | {'Gemini':<10}")
    print("-" * 50)
    print(f"{'P11 (Sens)':<20} | {p11_gpt:<10.4f} | {p11_gem:<10.4f}")
    print(f"{'P00 (Spec)':<20} | {p00_gpt:<10.4f} | {p00_gem:<10.4f}")
    print("-" * 50)
    print(f"{'Pc (Accuracy)':<20} | {pc_gpt:<10.4f} | {pc_gem:<10.4f}")
    print("="*50)

if __name__ == "__main__":
    main()

Ground Truth Statistics:
P1 (Prevalence of Hate Speech): 0.5000
P0 (Prevalence of Normal Text): 0.5000

--- Analysis for GPT ---
P11 (Sensitivity): 0.7950
P00 (Specificity): 0.7650
Pc (Accuracy):     0.7800

--- Analysis for Gemini ---
P11 (Sensitivity): 0.9400
P00 (Specificity): 0.7750
Pc (Accuracy):     0.8575

Metric               | GPT        | Gemini    
--------------------------------------------------
P11 (Sens)           | 0.7950     | 0.9400    
P00 (Spec)           | 0.7650     | 0.7750    
--------------------------------------------------
Pc (Accuracy)        | 0.7800     | 0.8575    


In [2]:
from google import genai
from google.genai import types

import pandas as pd
import json
import time
import os
import re
import glob
from tqdm.notebook import tqdm
from dotenv import load_dotenv

# =========================
# 1) SETUP & CREDENTIALS
# =========================
load_dotenv()
API_KEY = os.getenv("GEMINI_API_KEY")
if not API_KEY:
    raise ValueError("API Key not found! Check your .env file.")

# New SDK client + timeout (ms)
client = genai.Client(
    api_key=API_KEY,
    http_options=types.HttpOptions(timeout=60000)  # 60s
)

# =========================
# 2) CONFIGURATION
# =========================
INPUT_DIR = r"D:\PythonEnv\Interpretable\hs_sample_400\GPTResult_MIXED_NOISE"
OUTPUT_DIR_NAME = "gemini_res_3_0"

BATCH_SIZE = 10

# Primary / fallback models
PRIMARY_MODEL  = "gemini-3-flash-preview"
FALLBACK_MODEL = "gemini-2.5-flash-lite"  # very stable fallback

# Safety Settings (Allow hate speech detection)
SAFETY_SETTINGS = [
    types.SafetySetting(category="HARM_CATEGORY_HARASSMENT", threshold="BLOCK_NONE"),
    types.SafetySetting(category="HARM_CATEGORY_HATE_SPEECH", threshold="BLOCK_NONE"),
    types.SafetySetting(category="HARM_CATEGORY_SEXUALLY_EXPLICIT", threshold="BLOCK_NONE"),
    types.SafetySetting(category="HARM_CATEGORY_DANGEROUS_CONTENT", threshold="BLOCK_NONE"),
]

# =========================
# 3) PROMPT
# =========================
SYSTEM_PROMPT = (
    "You are a content moderation classifier for hate speech.\n"
    "Output JSON only.\n"
)

RULE_HINT = (
    "Binary classification for child-safety moderation. "
    "Set label=1 ONLY when the text clearly contains hate/derogatory/abusive/harassing/bullying/dehumanizing/threatening/"
    "violent content toward a person or group (explicit or strongly implied), including slurs, clear targeted insults, "
    "dehumanization, calls for exclusion/discrimination, or encouragement/celebration of harm. "
    "Set label=0 if it is unclear/ambiguous, non-targeted profanity, general rudeness without a clear target, sarcasm without "
    "clear abuse, or mere discussion/quoting without endorsement. "
    "Tie-break: if unsure, choose 0. "
)

# =========================
# 4) OPTIONAL: LIST MODELS
# =========================
def list_models_supporting_generate():
    print("=== Models that support generateContent ===")
    try:
        for m in client.models.list():
            actions = getattr(m, "supported_actions", None) or getattr(m, "supported_generation_methods", None) or []
            if "generateContent" in actions or "generate_content" in actions:
                print(m.name)
        print("==========================================")
    except Exception as e:
        print("list() failed:", e)

# list_models_supporting_generate()

# =========================
# 5) HELPERS
# =========================
def clean_json_response(response_text: str) -> str:
    """Sometimes model wraps JSON in ```json ... ```"""
    text = re.sub(r"```json\s*", "", response_text)
    text = re.sub(r"```\s*$", "", text)
    return text.strip()

def build_prompt(texts):
    """
    Request an ID->label mapping to avoid missing/shift issues.
    Output format:
      {"predictions": {"0":0, "1":1, ...}}
    """
    n = len(texts)
    lines = [
        SYSTEM_PROMPT.strip(),
        RULE_HINT.strip(),
        "",
        "Return JSON only in EXACT format: {\"predictions\": {\"0\":0, \"1\":1, ...}}",
        f"Constraints: predictions keys MUST be string IDs \"0\"..\"{n-1}\". "
        "If you cannot decide or the content is ambiguous, use 0. "
        "If an ID is missing, omit it (do NOT shift other IDs).",
        "",
        "Inputs:"
    ]
    for i, t in enumerate(texts):
        clean = str(t).replace("\n", " ")
        lines.append(f"{i}: {clean}")
    return "\n".join(lines)

def parse_predictions(data, n):
    """Expect: {"predictions": {"0":0, "1":1, ...}}. Missing -> 0."""
    if not isinstance(data, dict):
        raise ValueError("Response JSON is not an object.")
    preds_map = data.get("predictions", None)
    if not isinstance(preds_map, dict):
        raise ValueError("Missing/invalid 'predictions' mapping.")

    out = []
    for i in range(n):
        v = preds_map.get(str(i), 0)
        if v not in (0, 1):
            v = 0
        out.append(int(v))
    return out

def is_quota_error(e: Exception) -> bool:
    s = str(e).lower()
    return ("429" in s) or ("quota" in s) or ("rate limit" in s)

def is_blocked_error(e: Exception) -> bool:
    s = str(e).lower()
    return ("prohibited" in s) or ("blocked" in s) or ("safety" in s) or ("candidates" in s)

def get_text_parts_only(resp) -> str:
    """
    New SDK responses may contain non-text parts (e.g., thought_signature).
    Concatenate only actual text parts to stabilize JSON parsing.
    """
    cands = getattr(resp, "candidates", None) or []
    if not cands:
        return ""
    content = getattr(cands[0], "content", None)
    parts = getattr(content, "parts", None) or []
    texts = []
    for p in parts:
        t = getattr(p, "text", None)
        if t:
            texts.append(t)
    return "".join(texts).strip()

def generate_content_json(prompt: str, model_name: str):
    """One raw call with JSON mode."""
    return client.models.generate_content(
        model=model_name,
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0.0,
            response_mime_type="application/json",
            safety_settings=SAFETY_SETTINGS,
        ),
    )

# =========================
# 6) CLASSIFICATION
# =========================
def classify_batch_once(texts, max_retries=4):
    """
    Try PRIMARY model first; if it fails/empty, fall back to 2.5.
    Returns list[int] of len(texts), or raises (to let robust splitter handle).
    """
    n = len(texts)
    prompt = build_prompt(texts)

    for model_name in [PRIMARY_MODEL, FALLBACK_MODEL]:
        retries = 0
        while retries < max_retries:
            try:
                resp = generate_content_json(prompt, model_name)

                text = get_text_parts_only(resp)
                cands = getattr(resp, "candidates", None)

                # blocked/empty
                if (not text) or (cands is not None and len(cands) == 0):
                    raise RuntimeError("EMPTY_TEXT_OR_CANDIDATE")

                cleaned = clean_json_response(text)
                data = json.loads(cleaned)
                return parse_predictions(data, n)

            except Exception as e:
                # quota: wait and retry same model
                if is_quota_error(e):
                    wait_time = 5
                    print(f"\n⚠️ Quota hit! Sleeping {wait_time}s... (model={model_name})")
                    time.sleep(wait_time)
                    retries += 1
                    continue

                # blocked/empty: do not spin; let robust splitter isolate
                if is_blocked_error(e) or "empty_text_or_candidate" in str(e).lower():
                    raise RuntimeError(f"BLOCKED_OR_EMPTY: {e}")

                # other errors: quick retry a couple times, then move to fallback
                retries += 1
                if retries < max_retries:
                    time.sleep(0.5)
                    continue
                break  # switch to next model

    raise RuntimeError("BOTH_MODELS_FAILED")

def classify_texts_robust(texts, max_same_batch_retries=2):
    """
    Robust:
    - retry same batch a few times
    - if still fails, split into halves recursively
    - if single item still fails/blocked, default 0
    """
    if not texts:
        return []

    last_err = None
    for _ in range(max_same_batch_retries):
        try:
            return classify_batch_once(texts, max_retries=3)
        except Exception as e:
            last_err = e
            time.sleep(0.2)

    if len(texts) == 1:
        print(f"\n❌ Single item failed/blocked. Defaulting to 0. Error: {last_err}")
        return [0]

    mid = len(texts) // 2
    left = classify_texts_robust(texts[:mid], max_same_batch_retries=max_same_batch_retries)
    right = classify_texts_robust(texts[mid:], max_same_batch_retries=max_same_batch_retries)
    return left + right

# =========================
# 7) FILE PROCESSING
# =========================
def process_single_file(file_path, output_dir):
    filename = os.path.basename(file_path)
    print(f"\nProcessing file: {filename}")

    output_filename = filename.replace(".csv", "_gemini_result.csv")
    output_full_path = os.path.join(output_dir, output_filename)

    if os.path.exists(output_full_path):
        print(f"Skipping {filename} (Output already exists).")
        return

    try:
        df = pd.read_csv(file_path, encoding="utf-8-sig")
    except UnicodeDecodeError:
        df = pd.read_csv(file_path, encoding="latin1")

    if "text" not in df.columns:
        print(f"Skipping {filename} (No 'text' column found).")
        return

    results = []

    for i in tqdm(range(0, len(df), BATCH_SIZE), desc=f"Rows in {filename}", leave=False):
        batch = df.iloc[i : i + BATCH_SIZE]
        texts = batch["text"].tolist()

        preds = classify_texts_robust(texts)

        # Strong guarantee
        if len(preds) != len(texts):
            if len(preds) < len(texts):
                preds = preds + [0] * (len(texts) - len(preds))
            else:
                preds = preds[:len(texts)]

        results.extend(preds)

        # Keep tiny pause; quota sleep is handled separately
        time.sleep(0.05)

    df["hatespeech_LLM_gemini"] = results
    df.to_csv(output_full_path, index=False, encoding="utf-8-sig")
    print(f"✅ Saved: {output_filename}")

# =========================
# 8) MAIN
# =========================
def main():
    base_dir = os.path.dirname(INPUT_DIR)
    output_dir = os.path.join(base_dir, OUTPUT_DIR_NAME)

    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"Created output directory: {output_dir}")
    else:
        print(f"Output directory found: {output_dir}")

    search_pattern = os.path.join(INPUT_DIR, "traindataset_p_*.csv")
    all_files = glob.glob(search_pattern)
    all_files.sort()

    print(f"Primary model : {PRIMARY_MODEL}")
    print(f"Fallback model: {FALLBACK_MODEL}")
    print(f"Batch size    : {BATCH_SIZE}")
    print(f"Found {len(all_files)} files to process.")

    for file_path in tqdm(all_files, desc="Overall File Progress"):
        process_single_file(file_path, output_dir)

    print("\n" + "=" * 30)
    print("ALL FILES PROCESSED SUCCESSFULLY")
    print("=" * 30)

if __name__ == "__main__":
    main()


Output directory found: D:\PythonEnv\Interpretable\hs_sample_400\gemini_res_3_0
Model     : gemini-3-flash-preview
Batch size: 5
Found 11 files to process.


Overall File Progress:   0%|          | 0/11 [00:00<?, ?it/s]


Processing file: traindataset_p_00_sampled400_EXTREME_MIX_NOISE.csv


Rows in traindataset_p_00_sampled400_EXTREME_MIX_NOISE.csv:   0%|          | 0/80 [00:00<?, ?it/s]

[gemini-3-flash-preview] call ok in 6.38s | n=5 | has_text=True | cands=1
[gemini-3-flash-preview] call ok in 4.72s | n=5 | has_text=True | cands=1
[gemini-3-flash-preview] call ok in 9.63s | n=5 | has_text=True | cands=1
[gemini-3-flash-preview] call ok in 9.91s | n=5 | has_text=True | cands=1
[gemini-3-flash-preview] call ok in 4.34s | n=5 | has_text=True | cands=1
[gemini-3-flash-preview] call ok in 0.34s | n=5 | has_text=False | cands=0
[gemini-3-flash-preview] call fail in 0.34s | n=5 | err=RuntimeError: BLOCKED_OR_EMPTY_CANDIDATE
[gemini-3-flash-preview] call ok in 0.35s | n=5 | has_text=False | cands=0
[gemini-3-flash-preview] call fail in 0.35s | n=5 | err=RuntimeError: BLOCKED_OR_EMPTY_CANDIDATE
↘ split 5 -> 2+3 (err=BLOCKED_PROMPT: BLOCKED_OR_EMPTY_CANDIDATE)
[gemini-3-flash-preview] call ok in 50.36s | n=2 | has_text=True | cands=1
[gemini-3-flash-preview] call ok in 41.69s | n=3 | has_text=True | cands=1
[gemini-3-flash-preview] call ok in 27.11s | n=5 | has_text=True | can

KeyboardInterrupt: 

In [25]:
from google import genai
import os
from dotenv import load_dotenv

load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

print("List of models that support generateContent:\n")
for m in client.models.list():
    if "generateContent" in getattr(m, "supported_actions", []):
        print(m.name)


List of models that support generateContent:

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash-exp
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.0-flash-lite-preview-02-05
models/gemini-2.0-flash-lite-preview
models/gemini-exp-1206
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image-preview
models/gemini-2.5-flash-image
models/gemini-2.5-flash-preview-09-2025
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-robotics-er-1

In [24]:
!pip install -U google-genai


  Using cached websockets-15.0.1-cp39-cp39-win_amd64.whl.metadata (7.0 kB)
Using cached websockets-15.0.1-cp39-cp39-win_amd64.whl (176 kB)

   -------------------------- ------------- 2/3 [google-genai]
   -------------------------- ------------- 2/3 [google-genai]
   ---------------------------------------- 3/3 [google-genai]



In [2]:
import pandas as pd
import os
import glob
import re

# --- CONFIGURATION ---
# Path where the Gemini results are saved
RESULT_DIR = r"D:\PythonEnv\Interpretable\hs_sample_400\gemini_res_2_0"

def calculate_metrics_for_model(df, truth_col, pred_col, p1_truth, p0_truth):
    """
    Calculates Sensitivity (P11), Specificity (P00), and Accuracy (Pc)
    """
    if pred_col not in df.columns:
        return None, None, None

    # P11 = P(Pred=1 | Truth=1)
    truth_1 = df[df[truth_col] == 1]
    p11 = truth_1[pred_col].mean() if len(truth_1) > 0 else 0.0

    # P00 = P(Pred=0 | Truth=0)
    truth_0 = df[df[truth_col] == 0]
    p00 = (1.0 - truth_0[pred_col].mean()) if len(truth_0) > 0 else 0.0

    # Pc = P(Y=1)*P11 + P(Y=0)*P00
    pc = (p1_truth * p11) + (p0_truth * p00)

    return p11, p00, pc

def get_p_value_from_filename(filename):
    """
    Extracts the 'p_XX' part and converts to float (0.0 - 1.0)
    Assumes format: traindataset_p_05_...
    """
    match = re.search(r"_p_(\d+)_", filename)
    if match:
        num = int(match.group(1))
        return num / 10.0
    return -1

def main():
    if not os.path.exists(RESULT_DIR):
        print(f"Error: Directory not found: {RESULT_DIR}")
        return

    search_pattern = os.path.join(RESULT_DIR, "*_gemini_result.csv")
    files = glob.glob(search_pattern)

    if not files:
        print("No result files found. Make sure you ran the Gemini classifier script.")
        return

    summary_data = []
    print(f"Found {len(files)} files. Analyzing...")

    for file_path in files:
        filename = os.path.basename(file_path)

        try:
            df = pd.read_csv(file_path, encoding='utf-8-sig')
        except:
            df = pd.read_csv(file_path, encoding='latin1')

        p_val = get_p_value_from_filename(filename)

        if 'hs_state' not in df.columns:
            print(f"Skipping {filename}: 'hs_state' missing.")
            continue

        # Ground truth priors
        p1_truth = df['hs_state'].mean()
        p0_truth = 1.0 - p1_truth

        # Gemini metrics only
        gem_p11, gem_p00, gem_pc = calculate_metrics_for_model(
            df, 'hs_state', 'hatespeech_LLM_gemini', p1_truth, p0_truth
        )

        summary_data.append({
            "File_P": p_val,
            "Filename": filename,
            "Gemini_P11": gem_p11,
            "Gemini_P00": gem_p00,
            "Gemini_Pc": gem_pc
        })

    summary_df = pd.DataFrame(summary_data).sort_values(by="File_P")

    display_cols = ["File_P", "Gemini_P11", "Gemini_P00", "Gemini_Pc"]
    print("\n" + "="*70)
    print(f"{' GEMINI METRICS ':=^70}")
    print("="*70)
    print(summary_df[display_cols].round(4).to_string(index=False))

    output_path = os.path.join(RESULT_DIR, "summary_metrics_gemini.csv")
    summary_df.to_csv(output_path, index=False)
    print("\n" + "="*70)
    print(f"Summary saved to: {output_path}")

if __name__ == "__main__":
    main()


Found 11 files. Analyzing...

=========================== GEMINI METRICS ===========================
 File_P  Gemini_P11  Gemini_P00  Gemini_Pc
    0.0       0.815       0.705     0.7600
    0.1       0.715       0.735     0.7250
    0.2       0.665       0.760     0.7125
    0.3       0.610       0.745     0.6775
    0.4       0.540       0.755     0.6475
    0.5       0.505       0.820     0.6625
    0.6       0.410       0.820     0.6150
    0.7       0.400       0.860     0.6300
    0.8       0.315       0.755     0.5350
    0.9       0.260       0.860     0.5600
    1.0       0.380       0.835     0.6075

Summary saved to: D:\PythonEnv\Interpretable\hs_sample_400\gemini_res_2_0\summary_metrics_gemini.csv


In [1]:
import os, time
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()
API_KEY = os.getenv("GEMINI_API_KEY")
assert API_KEY, "No GEMINI_API_KEY in .env"

MODEL = "gemini-3-flash-preview"   # 你要测的模型
# MODEL = "gemini-2.5-flash-lite"  # 对照组

# 关键：加 timeout，避免无限卡住（单位：毫秒）
client = genai.Client(
    api_key=API_KEY,
    http_options=types.HttpOptions(timeout=60000)  # 60s
)

cfg = types.GenerateContentConfig(
    temperature=0.0,
    max_output_tokens=128,
    # 先别用 JSON mode，避免 schema/JSON 解析因素干扰；就测“是否有回复”
    # response_mime_type="application/json",
    safety_settings=[
        types.SafetySetting(category="HARM_CATEGORY_HARASSMENT", threshold="BLOCK_NONE"),
        types.SafetySetting(category="HARM_CATEGORY_HATE_SPEECH", threshold="BLOCK_NONE"),
        types.SafetySetting(category="HARM_CATEGORY_SEXUALLY_EXPLICIT", threshold="BLOCK_NONE"),
        types.SafetySetting(category="HARM_CATEGORY_DANGEROUS_CONTENT", threshold="BLOCK_NONE"),
    ],
)

def ping(prompt: str):
    print(f"\n=== PING model={MODEL} ===")
    print("Prompt:", prompt)
    t0 = time.time()
    try:
        resp = client.models.generate_content(
            model=MODEL,
            contents=prompt,
            config=cfg,
        )
        dt = time.time() - t0

        text = getattr(resp, "text", None)
        cands = getattr(resp, "candidates", None)

        print(f"Returned in {dt:.2f}s")
        print("Has text:", bool(text))
        print("Candidates:", 0 if cands is None else len(cands))

        # 打印一点点内容看看
        if text:
            print("Text (first 300 chars):", text[:300])
        else:
            print("No resp.text. Raw resp:", resp)

    except Exception as e:
        dt = time.time() - t0
        print(f"Exception after {dt:.2f}s: {type(e).__name__}: {e}")

# 1) 最简单文本
ping("Reply with exactly: OK")

# 2) 轻量 JSON（可选）
ping('Return JSON only: {"ok": true}')



=== PING model=gemini-3-flash-preview ===
Prompt: Reply with exactly: OK


Returned in 0.96s
Has text: True
Candidates: 1
Text (first 300 chars): OK

=== PING model=gemini-3-flash-preview ===
Prompt: Return JSON only: {"ok": true}


Returned in 1.05s
Has text: True
Candidates: 1
Text (first 300 chars): {"ok": true}
